AUTHOR : GLEN OCHIENG

REG NO : C004/600555/2026

# Machine Learning Assignment - Customer Churn Prediction

Use appropriate machine learning algorithm/model to demonstrate how machine learning can be used to detect customer churn (customers who may be souring towards the company). 

## Information on the Dataset

For this project I will be using the IBM Telco Customer Churn Dataset available at: https://data.mendeley.com/datasets/phsxg9ssrf/1 

Alternatively, it can be uploaded directly to Jupyter using the direct link : https://raw.githubusercontent.com/Glen-Ochieng/Customer-Churn-Machine-Learning-Application/refs/heads/main/Telco_customer_churn_dataset.csv


* Each row represents a customer.
* Columns describe customer metadata and whether they have churned (left or stayed).


In [1]:
# loading in necessary libraries 
import pandas as pd 
import numpy as np

In [2]:
# For reproducibility 
np.random.seed(42)

In [3]:
# Loading in our dataset 
url = 'https://raw.githubusercontent.com/Glen-Ochieng/Customer-Churn-Machine-Learning-Application/refs/heads/main/Telco_customer_churn_dataset.csv'
data = pd.read_csv(url)
data.head(2)

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved


## Exploratory Data Analysis 

In [4]:
# Checking the dimensions of our dataset
data.shape

(7043, 33)

In [5]:
# Checking the data types of each column
data.dtypes

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges            str
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object

From inspection, Total charges is recorded as a string yet it should be a numeric!

In [6]:
# Converting 'Total Charges' from a string to a numeric 
data['Total Charges'] = pd.to_numeric(data['Total Charges'], errors = 'coerce')
data['Total Charges'].dtypes

dtype('float64')

In [8]:
# Checking if the dataset has null values 
missing = data.isnull().sum()
print(missing[missing>0])

Total Charges      11
Churn Reason     5174
dtype: int64


So the two columns that have nulls are the 'Total Charges' column and the 'Churn Reason' column.

* For the 'Total Charges' column, the corresponding 11 entries with empty Total Charges values can be dropped.

* However, using intuition, we can hypothesize that the nulls in the 'Churn Reason' column could be the customers who did not churn. 

In [9]:
# checking if the number of customers who did not churn matches the nulls (5174)
data['Churn Label'].value_counts()

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

Voila! We have confirmed it.

The customers who churned are **1,869** and those that did not are **5,174**.

In [10]:
# dropping the nulls from the 'Total Charges' columns
data = data.dropna(subset = ['Total Charges'])

In [11]:
# Checking that our dependent variable (Churn Label) is binary
print(data['Churn Label'].unique())

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


In [12]:
# Checking the statistics of key numeric columns
data.describe().round(2)

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV
count,7032.0,7032.00,7032.00,7032.00,7032.00,7032.00,7032.00,7032.00,7032.00,7032.00
mean,1.0,93523.03,36.28,-119.80,32.42,64.80,2283.30,0.27,58.72,4401.45
std,0.0,1865.52,2.46,2.16,24.55,30.09,2266.77,0.44,21.53,1182.41
min,1.0,90001.00,32.56,-124.30,1.00,18.25,18.80,0.00,5.00,2003.00
25%,1.0,92102.00,34.03,-121.82,9.00,35.59,401.45,0.00,40.00,3469.75
50%,1.0,93552.50,36.39,-119.74,29.00,70.35,1397.48,0.00,61.00,4527.50
75%,1.0,95354.00,38.23,-118.04,55.00,89.86,3794.74,1.00,75.00,5381.00
max,1.0,96161.00,41.96,-114.19,72.00,118.75,8684.80,1.00,100.00,6500.00


Notice the average monthly charge is about **65 dollars**.

The highest amount charged to a customer is 118.75 and the minimum is 18.25. A quarter of the customers have relatively low monthly charges, that is below 35.59, half of the customers have a monthly charge below 70.35 and three-quarters of the customers have monthly charges below 89.86.

The average tenure is about **32 months**  which is around 2.7 years. The standard deviation is relatively high (24.55), suggesting that there is a wide variation in customer tenure. This could indicate that there are both long-term and short-term customers in the dataset. The median value (50th percentile) of 29 months suggests that half of the customers have been with the company for less than 2.5 years. The maximum tenure of 72 months suggests the presence of long-term customers who have been with the company for the entire duration of the dataset.


## Model Pre-processing

We can now move on to dropping unnecessary columns for modelling.

* Since 'CustomerID' and 'Count' are just identification variables , they have no impact on customer churn, we can drop them.

* We should also drop the location columns other than longitude and latitude as they are all correlated to each other.

* We can also drop 'Churn Reason' as it only applies to the customers that have already churned and not the entire sample size.

* Consequently, we can drop 'Churn Label' as it is Yes for churned and No for not, which is the same information as 'Churn Value' which has 1 for churned and 0 for not churned.

* Finally, we should drop 'Churn Score' as it is a pre-calculated metric that will introduce bias to our model training.

In [13]:
model_data = data.drop(columns=['CustomerID', 'Count', 'Churn Reason',
                                'Country', 'State', 'City', 'Zip Code',
                                'Lat Long', 'Churn Label', 'Churn Score'])

In [14]:
# Creating the dependent and independent variables 

X = model_data.drop(columns=['Churn Value'])
y = model_data['Churn Value']

In [15]:
X.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Latitude           7032 non-null   float64
 1   Longitude          7032 non-null   float64
 2   Gender             7032 non-null   str    
 3   Senior Citizen     7032 non-null   str    
 4   Partner            7032 non-null   str    
 5   Dependents         7032 non-null   str    
 6   Tenure Months      7032 non-null   int64  
 7   Phone Service      7032 non-null   str    
 8   Multiple Lines     7032 non-null   str    
 9   Internet Service   7032 non-null   str    
 10  Online Security    7032 non-null   str    
 11  Online Backup      7032 non-null   str    
 12  Device Protection  7032 non-null   str    
 13  Tech Support       7032 non-null   str    
 14  Streaming TV       7032 non-null   str    
 15  Streaming Movies   7032 non-null   str    
 16  Contract           7032 non-null   str  

In [16]:
# Converting the datatypes of the independent variables from strings via one-hot encoding
X = pd.get_dummies(X, drop_first = True)

# drop_first reduces duplicate columns that are correlated i.e. instead of Gender_Male and Gender_Female we have just Gender_Male with True or False

X.head(2)

,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,CLTV,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,33.964131,-118.272783,2,53.85,108.15,3239,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
1,34.059281,-118.307420,2,70.70,151.65,2701,False,False,False,True,...,False,False,False,False,False,False,True,False,True,False


### Splitting the Data for Training and Testing

In [17]:
# Assuming scikit_learn is already installed on your current version of Python if not, please run
# %pip install scikit_learn

In [18]:
# loading in necessary packages

from sklearn.model_selection import train_test_split

In [19]:
# Splitting 70% to train and 30% to test
# Setting random_state for reproducibility
# stratify ensures that randomization does not again make the two classes uneven during the split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.7, random_state = 42, stratify = y)

In [20]:
X_train.shape,X_test.shape, y_train.shape, y_test.shape

((4922, 33), (2110, 33), (4922,), (2110,))

In [21]:
# Checking that both sets keep the natural class balance (about 27% churn)
print(y_train.value_counts(normalize = True).round(3))
print()
print(y_test.value_counts(normalize = True).round(3))

Churn Value
0    0.734
1    0.266
Name: proportion, dtype: float64

Churn Value
0    0.734
1    0.266
Name: proportion, dtype: float64


### Scaling

In [22]:
X_train.describe().loc[['min','max','std']].round(2)

,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,CLTV
min,32.56,-124.30,1.00,18.40,18.80,2003.00
max,41.96,-114.19,72.00,118.60,8684.80,6500.00
std,2.46,2.16,24.59,30.26,2277.87,1179.04


Notice the high spread of values in the columns : Tenure Months, Monthly Charges, Total Charges and CLTV as compared to the other columns: Latitude, Longitude.

These larger-range features usually dominate distance-based or gradient-based calculations such as those involved in Logistic Regression, SVMs, k-NN and neural networks. This is because these models rely on distance or optimization steps that assume features are on comparable scales.

Hence the need for scaling.

Scaling ensures each feature contributes proportionally, rather than being overshadowed by features with larger numeric values.

**Rules for Scaling**

* Must be done on train data only! So ensure you have split before scaling!

* Must be done on only the numeric variables of the train data, not the dummy variables!

In [23]:
# loading necessary packages
from sklearn.preprocessing import StandardScaler

In [24]:
# Listing the numeric columns that I will be applying scaling to
num_cols = ['Latitude', 'Longitude', 'Tenure Months',
            'Monthly Charges', 'Total Charges', 'CLTV']

In [25]:
# creating a copy so the original data is not lost after scaling, and can be used in later stages

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

In [26]:
# Applying the scaling to the numeric columns 

scaler = StandardScaler()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [27]:
#Checking on the impact of scaling on the training dataset
X_train_scaled.describe().loc[['min','max','std']]

,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,CLTV
min,-1.509920,-2.091243,-1.280909,-1.535103,-0.998019,-2.045202
max,2.314091,2.590541,1.607113,1.776770,2.806797,1.769309
std,1.000102,1.000102,1.000102,1.000102,1.000102,1.000102


In [28]:
#Checking on the impact of scaling on the test dataset
X_test_scaled.describe().loc[['min','max','std']]

,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,CLTV
min,-1.509920,-2.052341,-1.280909,-1.540061,-0.997997,-2.039265
max,2.314091,2.590541,1.607113,1.781728,2.801374,1.768461
std,0.994887,0.997511,0.994666,0.981267,0.983945,1.009405


And now, all the variables have smaller ranges and thus less disproportional impact.

## Modeling 

There are an array of classification models that can be used to predict customer churn. For this particular analysis, I will put the below models to the test to find out which is the best for our current dataset.

1. Logistic Regression
2. Random Forest 
3. Gradient Boosting
4. Support Vector Machine
5. Gaussian Naive Bayes
6. k-Nearest Neighbours
7. Decision Trees
8. XGBoost
9. LightGBM

In [29]:
# installing these libraries - xgboost and lightgbm
import importlib.util

# Installing xgboost and lightgbm only if they are not already installed
missing = [p for p in ["xgboost", "lightgbm"] if importlib.util.find_spec(p) is None]

if missing:
    %pip install -q --disable-pip-version-check {' '.join(missing)}

In [30]:
# loading in the model libraries 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [31]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

#### Balancing

Since our dataset has more customers who did not churn than who did, the model tends to favour the majority class and under-represent churners.

For the XGBoost model, I will place a manual balancing formula.

Other techniques such as SMOTE(Synthetic Minority Oversampling Technique) could have been alternatively applied.

In [32]:
# Ratio of majority to minority rows, used by XGBoost (assumes 0 is the majority class)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(max_iter=10000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(class_weight="balanced", random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "k Nearest Neighbours": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42),
    "LightGBM": LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1),
}

rows = []

for model_name, m in models.items():
    # Fitting the model to the scaled training data (features and labels come as a pair)
    m.fit(X_train_scaled, y_train)

    # Predicting on the TEST data
    y_pred = m.predict(X_test_scaled)

    # AUC ROC needs a score, not a hard 0 or 1 prediction
    if hasattr(m, "predict_proba"):
        y_score = m.predict_proba(X_test_scaled)[:, 1]
    else:
        y_score = m.decision_function(X_test_scaled)   # used by SVC

    rows.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_score),
    })

results_df = pd.DataFrame(rows)
results_df.round(3)

,Model,Accuracy,Precision,Recall,F1 Score,AUC-ROC
0,Logistic Regression,0.743,0.510,0.795,0.622,0.845
1,Random Forest,0.784,0.583,0.665,0.621,0.835
2,Gradient Boosting,0.799,0.650,0.526,0.581,0.845
3,SVM,0.735,0.501,0.797,0.615,0.832
4,Gaussian Naive Bayes,0.677,0.444,0.856,0.585,0.815
5,k Nearest Neighbours,0.772,0.568,0.592,0.579,0.779
6,Decision Tree,0.731,0.496,0.733,0.592,0.824
7,XGBoost,0.767,0.554,0.642,0.595,0.827
8,LightGBM,0.768,0.548,0.738,0.629,0.839


Because only about 27% of customers churned, accuracy can be misleading, so models were compared mainly on F1 score for the churn class, with AUC ROC as a supporting measure. 

The top 3 models are:
1. LightGBM (0.629)
2. Logistic Regression (0.622)
3. Random Forest (0.621)

### Hyperparameter Tuning 

Most of our models were running on their default parameters, which is not ideal for the very best models. We tried setting some parameters earlier with max_iter = 10000 for the Logistic Regression Model and max_depth = 6 for the decision tree. But now for the final three models, we can use hyperparameter tuning to see if any difference among the three models will appear.

For this assignment, grid search will be used. This is where you define a grid of the possible hyperparameters values and evaluate the model's performance for each combination. This ensures that all the possible combinations have been explored.

**Hyperparameter Tuning is very taxing on your computer's RAM, when it feels like it has run forever just be patient**

#### Hyperparameter Tuning for The LightGBM model

In [33]:
# importing the necessary library
from sklearn.model_selection import GridSearchCV

In [34]:
# Defining the parameter grid for LightGBM
param_grid_lgbm = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15],
    'learning_rate': [0.1, 0.01, 0.001]
}

In [35]:
# Creating a LightGBM classifier
lgbm = LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)

# Performing Grid Search Cross Validation
grid_search_lgbm = GridSearchCV(lgbm, param_grid_lgbm, scoring='f1', cv=5)
grid_search_lgbm.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMClassifie...2, verbose=-1)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.01, ...], 'max_depth': [5, 10, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=

In [36]:
# Getting the best parameters and best F1 score
best_params_lgbm = grid_search_lgbm.best_params_
best_score_lgbm = grid_search_lgbm.best_score_

In [37]:
#Printing the best hyperparameters
print(f"The best hyperparameters for the light gbm model are:\n{best_params_lgbm}")

The best hyperparameters for the light gbm model are:
{'learning_rate': 0.01, 'max_depth': 15, 'n_estimators': 300}


#### Hyperparameter Tuning for The Logistic Regression model

In [38]:
# Defining the parameter grid for Logistic Regression
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0, 1],
    'solver': ['liblinear', 'saga'],
    'class_weight': [None, 'balanced']
}

In [39]:
# Creating a Logistic Regression classifier
lr = LogisticRegression(max_iter=10000, random_state=42)

# Performing Grid Search Cross Validation
grid_search_lr = GridSearchCV(lr, param_grid_lr, scoring='f1', cv=5)
grid_search_lr.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.001, 0.01, ...], 'class_weight': [None, 'balanced'], 'l1_ratio': [0, 1], 'solver': ['liblinear', 'saga']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbo

In [40]:
# Getting the best parameters and best F1 score
best_params_lr = grid_search_lr.best_params_
best_score_lr = grid_search_lr.best_score_

In [42]:
#Printing the best hyperparameters
print(f"The best hyperparameters for the logistic model are:\n{best_params_lr}")

The best hyperparameters for the logistic model are:
{'C': 0.1, 'class_weight': 'balanced', 'l1_ratio': 0, 'solver': 'liblinear'}


#### Hyperparameter Tuning for the Random Forest Model

In [43]:
RandomForestClassifier().get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [44]:
# Defining the parameter grid for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5,10],
    'min_samples_leaf': [1, 2,4]
}

In [45]:
# loading in the necessary library
from sklearn.model_selection import GridSearchCV

In [47]:
# Defining a random forest model
rf = RandomForestClassifier(class_weight='balanced', random_state=42)

#Performing a grid search using cross validation
grid_search_rf = GridSearchCV(rf, param_grid_rf, scoring='f1', cv=5)
grid_search_rf.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True

In [48]:
# Getting the best parameters
best_params_rf = grid_search_rf.best_params_
best_score_rf = grid_search_rf.best_score_

In [49]:
#Displaying the best parameters
print(f"The best hyperparameters for the Random Forest model are:\n{best_params_rf}")

The best hyperparameters for the Random Forest model are:
{'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}


#### Performance Comparison after Hyperparameter Tuning

In [50]:
# Defining the models with the best hyperparameters
tuned_models = {
    'LightGBM (Tuned)': LGBMClassifier(
        **best_params_lgbm, class_weight='balanced',
        random_state=42, verbose=-1),
    'Logistic (Tuned)': LogisticRegression(
        **best_params_lr, max_iter=10000, random_state=42),
    'Random Forest (Tuned)': RandomForestClassifier(
        **best_params_rf, class_weight='balanced', random_state=42)}

rows = []

for model_name, m in tuned_models.items():
    # Fitting the model to the scaled training data (features and labels come as a pair)
    m.fit(X_train_scaled, y_train)

    # Predicting on the TEST data
    y_pred = m.predict(X_test_scaled)
    
    # AUC ROC needs a probability score, calculated for the current model
    y_score = m.predict_proba(X_test_scaled)[:, 1]

    rows.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_score),
    })

results_df = pd.DataFrame(rows)
results_df.round(3)


,Model,Accuracy,Precision,Recall,F1 Score,AUC-ROC
0,LightGBM (Tuned),0.748,0.518,0.775,0.621,0.845
1,Logistic (Tuned),0.743,0.511,0.766,0.613,0.845
2,Random Forest (Tuned),0.763,0.536,0.790,0.639,0.847


The leading model in terms of F1-score is the tuned Random Forest model, with an F1 score of **0.639.**

### Feature Importance

Once a final model has been chosen, we can refine it better by focusing on only the most influential variables.

This can be achieved using **Feature Importance**.

Feature importance is a technique used to identify the most influential features or variables in predicting the target variable. It helps us understand which features have the most significant impact on the model's predictions and provides insights into the underlying relationships between the features and the target variable.

In [51]:
# Extracting the tuned Random Forest
rf_model_tuned = tuned_models['Random Forest (Tuned)']

# Getting the feature importances from the Random Forest model
importances = rf_model_tuned.feature_importances_

# Creating a dataframe to store the feature importances
feature_importance_df = pd.DataFrame({
    'Feature': X_test_scaled.columns,
    'Importance': importances
})

# Sorting the dataframe by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False, ignore_index=True)

feature_importance_df.round(4)

,Feature,Importance
0,Tenure Months,0.1479
1,Total Charges,0.1030
2,Contract_Two year,0.0945
3,Dependents_Yes,0.0830
4,Monthly Charges,0.0685
5,Internet Service_Fiber optic,0.0632
6,Payment Method_Electronic check,0.0447
7,CLTV,0.0418
8,Latitude,0.0375
9,Longitude,0.0371


We can now step by step start removing the least important variables

#### Tuned Random Forest model without Phone Service_Yes

Recall the tuned model with Phone Service had an F1 Score of **0.639**

In [52]:
#Dropping the Phone Service column in both the training and test datasets
X_train_scaled_no_ps = X_train_scaled.drop(columns = ['Phone Service_Yes'])
X_test_scaled_no_ps = X_test_scaled.drop(columns =['Phone Service_Yes'])

In [53]:
#Fitting the model in the train dataset
rf_model_tuned.fit(X_train_scaled_no_ps,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [54]:
#Predicting on the test dataset
y_pred_no_ps= rf_model_tuned.predict(X_test_scaled_no_ps)

In [55]:
#Evaluating the model without Phone Service
f1_rf = f1_score(y_test, y_pred_no_ps)
accuracy_rf = accuracy_score(y_test,y_pred_no_ps)

In [58]:
print(f'The f1 score for the tuned Random Forest model without Phone Service column is: {f1_rf}')
print(f'\n The accuracy score for the tuned Random Forest model without Phone Service column is: {accuracy_rf}')

The f1 score for the tuned Random Forest model without Phone Service column is: 0.6398852223816356

 The accuracy score for the tuned Random Forest model without Phone Service column is: 0.762085308056872


#### Observation

Dropping the Phone Service column had no meaningful impact on our model scores. 

Models are encouraged to be parsimonious, and dropping columns such as Multiple Lines and Phone Service would achieve that. However, they are useful metadata, and since removing Phone Service had no real impact, I will keep it in the final model.

### Final Model 

In [59]:
rf_model_tuned 

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [60]:
# Predicting using final model
rf_model_tuned.fit(X_train_scaled, y_train)
y_pred = rf_model_tuned.predict(X_test_scaled) 

In [61]:
from sklearn.metrics import classification_report, confusion_matrix

In [62]:
print(confusion_matrix(y_test,y_pred))

[[1166  383]
 [ 118  443]]


In [64]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.91      0.75      0.82      1549
           1       0.54      0.79      0.64       561

    accuracy                           0.76      2110
   macro avg       0.72      0.77      0.73      2110
weighted avg       0.81      0.76      0.77      2110



In [65]:
rf_model_tuned.score(X_test_scaled,y_test)

0.7625592417061612

Overall, our chosen model scores well with an accuracy of **0.76**.